# Madhava Series
#### Accompanies Section 1.3 of *Experimental Mathematics: A Computational Perspective* by Matthew P. Richey and Matthew L. Wright

## Computing a Partial Sum
In this notebook we compute approximations to $\pi$ using partial sums of the Taylor series for $\arctan(x)$. Specifically, for the infinite series:
$$ \frac{\pi}{4}=1-\frac{1}{3}+\frac{1}{5}-\frac{1}{7}+\cdots=\sum_{k=1}^\infty \frac{(-1)^{k+1}}{2k-1} $$

the partial sum is a finite sum:

$$ \frac{\pi}{4} \approx 1-\frac{1}{3}+\frac{1}{5}-\frac{1}{7}+\cdots=\sum_{k=1}^m\frac{(-1)^{k+1}}{2k-1} $$

for some maximum value $m$.

To compute this partial sum, we add up the first $m$ terms of this series using a loop with an accumulator. Our accumulator, the variable `acc` in the code below, stores the running total as we add up the terms.

In [ ]:
from fractions import Fraction

m = 50 # number of terms to add up
acc = Fraction(0, 1) # accumulator, initialized to zero

# main loop
for i in range(1, m + 1):
    # Uncomment the following line to print out the values that we add to the accumulator.
    # Remember that print statements inside of a loop are helpful for debugging!
    # print(Fraction((-1)**(i + 1), 2*i - 1))

    acc += Fraction((-1)**(i + 1), 2*i - 1)

The value `4 * acc` is our approximation of $\pi$. If we simply print this number, we'll get a big exact fraction:

In [ ]:
4 * acc

We can also convert the result to a decimal number with any desired amount of digits. Here, we'll ask for 20 digits:

In [ ]:
import mpmath

mpmath.mp.dps = 20
mpf_num = mpmath.mpf(4*acc.numerator)/mpmath.mpf(acc.denominator)
mpmath.nstr(mpf_num, 20)

## Analyzing Accuracy
Let's find out how the accuracy of the approximation depends on the number of terms and recreate Table 1.2 in the text. For this, it is convenient to define a function (a `def` in Python) that returns the $m$-term partial sum of the Madhava series.

It's important to consider the number type that the computer will use for this calculation. In the previous calculation we used fractions (exact values), but this may be rather slow if we want to add a large number of terms. The fastest option is to use floating-point numbers throughout the computation, but this provides a maximum of about 16 decimal digits and suffers from round-off error in repeated addition operations. A good middle ground is to use a large, but fixed, number of decimal digits throughout the calculation.

In the code below, we use the `mpmath` library (similar to Mathematica's `N` function) to specify that numbers are stored with 100 decimal digits of precision. This code is similar to Pseudocode 1.19 in the text.

In [ ]:
def madhavaSum(m):
    mpmath.mp.dps = 100 # note 100 decimal digits
    acc = mpmath.mpf(0) # initialize the accumulator as a local variable

    for i in range(1, m + 1):
        acc += mpmath.mpf((-1)**(i + 1)) / (2*i - 1)

    return 4 * acc

Try it out, computing the first ten partial sums. Python's list comprehension provides a convenient way to do this.

In [ ]:
[madhavaSum(n) for n in range(1, 11)]

Now we will compute the partial sums for each value of `m`. We'll print each partial sum to 20 decimal digits.

In [ ]:
mVals = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]

piVals = [madhavaSum(m) for m in mVals]
[mpmath.nstr(val, 20) for val in piVals]

Now we can compute the accuracy of each approximation using the $\log_{10}$ method described in the text.

In [ ]:
accuracy = []
for val in piVals:
    diff = abs(mpmath.pi - val)
    accuracy.append(float(-mpmath.log10(diff)))

[round(acc, 3) for acc in accuracy]

We can display these values in a table as follows:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'm': mVals,
    'approximation': [mpmath.nstr(val, 20) for val in piVals],
    'accuracy': [round(acc, 3) for acc in accuracy]
})
df

What is the shape of the accuracy plot? As in the text, we try to transform the accuracy values to make them linear. We will make plots similar to those in Figure 1.6 in the text. First, we try squaring the accuracy values, which is easy to do as follows.

In [ ]:
import matplotlib.pyplot as plt

accuracy_squared = [acc**2 for acc in accuracy]

plt.figure(figsize=(6, 4))
plt.plot(mVals, accuracy_squared, marker='o')
plt.xlabel('number of terms')
plt.ylabel('accuracy^2')
plt.grid(True)
plt.show()

The squared values are still not linear. Instead, let's try raising 10 to the power of each accuracy value, as follows.

In [ ]:
accuracy_10_power = [10**acc for acc in accuracy]

plt.figure(figsize=(6, 4))
plt.plot(mVals, accuracy_10_power, marker='o')
plt.xlabel('number of terms')
plt.ylabel('10^accuracy')
plt.grid(True)
plt.show()